# Noah-MP East River WY2023 NARR point forcing

Start small: extract only the nearest NARR grid cell to Gothic / East River for water year 2023 (2022-10-01 00 UTC through 2023-09-30 21 UTC).

This avoids keeping full NARR grids on scratch. On NCAR systems the notebook reads `NARRflx` tar files directly from the local RDA archive. Off NCAR, it can download one tar at a time, extract the point values, and optionally delete the tar cache afterward.


In [ ]:
from pathlib import Path
import calendar
import os
import shutil
import subprocess
import tarfile
import tempfile

import numpy as np
import pandas as pd

from eccodes import (
    codes_get,
    codes_get_array,
    codes_grib_new_from_file,
    codes_release,
)

os.environ.setdefault("MPLCONFIGDIR", "/glade/derecho/scratch/cdalden/tmp/matplotlib")

TARGET_NAME = "gothic_east_river"
TARGET_LAT = 38.96
TARGET_LON = -106.99

WY = 2023
START = pd.Timestamp("2022-10-01 00:00")
END = pd.Timestamp("2023-09-30 21:00")

RDA_ROOT = Path("/glade/campaign/collections/rda/data/d608000")
WORK_ROOT = Path("/glade/derecho/scratch/cdalden/tmp/noahmp_east_river_wy2023")
TAR_CACHE_DIR = WORK_ROOT / "narrflx_tar_cache"
FORCING_DIR = WORK_ROOT / "forcing"
RUN_DIR = WORK_ROOT / "hrldas_run"

for path in [TAR_CACHE_DIR, FORCING_DIR, RUN_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(WORK_ROOT)


## Build the WY2023 NARRflx tar list

NARR `3HRLY` flux files are four tar files per month: days 01-08, 09-16, 17-24, and 25-end-of-month. Each tar contains 3-hourly `merged_AWIP32.YYYYMMDDHH.RS.flx` GRIB1 files.


In [ ]:
def month_chunks(year, month):
    last_day = calendar.monthrange(year, month)[1]
    return [f"{start:02d}{end:02d}" for start, end in [(1, 8), (9, 16), (17, 24), (25, last_day)]]


def narrflx_tar_table(start=START, end=END):
    rows = []
    for period in pd.period_range(start.to_period("M"), end.to_period("M"), freq="M"):
        year = period.year
        month = period.month
        yyyymm = f"{year}{month:02d}"
        for chunk in month_chunks(year, month):
            name = f"NARRflx_{yyyymm}_{chunk}.tar"
            rows.append(
                {
                    "year": year,
                    "month": month,
                    "chunk": chunk,
                    "name": name,
                    "rda_path": RDA_ROOT / "3HRLY" / str(year) / name,
                    "cache_path": TAR_CACHE_DIR / name,
                    "url": f"https://data.rda.ucar.edu/d608000/3HRLY/{year}/{name}",
                }
            )
    return pd.DataFrame(rows)

tar_table = narrflx_tar_table()
tar_table.head(), tar_table.tail(), len(tar_table)


## Access one NARRflx tar at a time

Default behavior uses the NCAR local RDA archive and does not copy tar files. If the local archive is unavailable, set `ALLOW_NETWORK_DOWNLOAD = True`; each tar is cached only long enough to extract the point forcing unless `KEEP_DOWNLOADED_TARS = True`.


In [ ]:
ALLOW_NETWORK_DOWNLOAD = False
KEEP_DOWNLOADED_TARS = False

# Use None for the full water year. Set to 1 while debugging.
POINT_FIRST_N_TARS = None
point_tars = tar_table if POINT_FIRST_N_TARS is None else tar_table.head(POINT_FIRST_N_TARS)


def get_tar_path(row, allow_network=ALLOW_NETWORK_DOWNLOAD):
    rda_path = Path(row.rda_path)
    cache_path = Path(row.cache_path)
    if rda_path.exists():
        return rda_path, False
    if cache_path.exists() and cache_path.stat().st_size > 0:
        return cache_path, True
    if not allow_network:
        raise FileNotFoundError(
            f"Missing local RDA file {rda_path}. Set ALLOW_NETWORK_DOWNLOAD=True to download {row.url}"
        )

    import requests

    tmp = cache_path.with_suffix(cache_path.suffix + ".part")
    with requests.get(row.url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with tmp.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    tmp.rename(cache_path)
    return cache_path, True

check = []
for row in point_tars.head(5).itertuples(index=False):
    path = Path(row.rda_path)
    check.append({"name": row.name, "local_rda_exists": path.exists(), "size_gb": path.stat().st_size / 1024**3 if path.exists() else np.nan})
pd.DataFrame(check)


## GRIB field mapping for the Gothic point

These GRIB1 files use a local NARR parameter table, so text names can be misleading. The extractor uses parameter IDs plus level metadata. The first six fields are the main atmospheric drivers for a point Noah-MP test; radiation/precipitation IDs should be confirmed against your target HRLDAS forcing convention before a production run.


In [ ]:
FIELD_SPECS = {
    "T2_K": {"param": 11, "typeOfLevel": "heightAboveGround", "level": 2},
    "Q2_kgkg": {"param": 51, "typeOfLevel": "heightAboveGround", "level": 2},
    "RH2_pct": {"param": 52, "typeOfLevel": "heightAboveGround", "level": 2},
    "U10_ms": {"param": 33, "typeOfLevel": "heightAboveGround", "level": 10},
    "V10_ms": {"param": 34, "typeOfLevel": "heightAboveGround", "level": 10},
    "PSFC_Pa_or_pressure_field": {"param": 1, "typeOfLevel": "hybrid", "level": 1},
    "T10_K": {"param": 11, "typeOfLevel": "heightAboveGround", "level": 10},
    "Q10_kgkg": {"param": 51, "typeOfLevel": "heightAboveGround", "level": 10},
    "SOIL_T_K_layer0": {"param": 85, "typeOfLevel": "depthBelowLandLayer", "level": 0},
    "SOIL_M_layer0": {"param": 86, "typeOfLevel": "depthBelowLandLayer", "level": 0},
    "RAD_param211_nominalTop": {"param": 211, "typeOfLevel": "nominalTop", "level": 0},
    "RAD_param212_nominalTop": {"param": 212, "typeOfLevel": "nominalTop", "level": 0},
}


def timestamp_from_grib_member(member_name):
    stamp = Path(member_name).name.split(".")[1]
    return pd.Timestamp.strptime(stamp, "%Y%m%d%H")


def grib_inventory(path):
    rows = []
    with Path(path).open("rb") as f:
        i = 0
        while True:
            gid = codes_grib_new_from_file(f)
            if gid is None:
                break
            i += 1
            try:
                rows.append(
                    {
                        "message": i,
                        "param": codes_get(gid, "indicatorOfParameter"),
                        "typeOfLevel": codes_get(gid, "typeOfLevel"),
                        "level": codes_get(gid, "level"),
                        "stepRange": codes_get(gid, "stepRange"),
                        "name": codes_get(gid, "name"),
                        "shortName": codes_get(gid, "shortName"),
                        "units": codes_get(gid, "units"),
                    }
                )
            finally:
                codes_release(gid)
    return pd.DataFrame(rows)


## Extract WY2023 Gothic point forcing

This cell writes one temporary GRIB file at a time under `/tmp`, extracts the nearest NARR point, and removes the temporary file when the cell exits. The persistent output is only a small CSV.


In [ ]:
def nearest_values_from_grib_file(grib_path, target_lat=TARGET_LAT, target_lon=TARGET_LON):
    wanted = {(spec["param"], spec["typeOfLevel"], spec["level"]): name for name, spec in FIELD_SPECS.items()}
    row = {name: np.nan for name in FIELD_SPECS}
    row.update({"narr_lat": np.nan, "narr_lon": np.nan})

    with Path(grib_path).open("rb") as f:
        while True:
            gid = codes_grib_new_from_file(f)
            if gid is None:
                break
            try:
                key = (
                    codes_get(gid, "indicatorOfParameter"),
                    codes_get(gid, "typeOfLevel"),
                    codes_get(gid, "level"),
                )
                name = wanted.get(key)
                if name is None:
                    continue
                lats = codes_get_array(gid, "latitudes")
                lons = codes_get_array(gid, "longitudes")
                vals = codes_get_array(gid, "values")
                lons = ((lons + 180) % 360) - 180
                idx = int(np.nanargmin((lats - target_lat) ** 2 + (lons - target_lon) ** 2))
                row[name] = float(vals[idx])
                lat_i = float(lats[idx])
                lon_i = float(lons[idx])
                if abs(lat_i - target_lat) < 1.0 and abs(lon_i - target_lon) < 1.0:
                    row["narr_lat"] = lat_i
                    row["narr_lon"] = lon_i
            finally:
                codes_release(gid)
    return row


def extract_point_from_tar(tar_path, tmp_dir):
    rows = []
    with tarfile.open(tar_path) as tf:
        members = sorted(m for m in tf.getmembers() if m.isfile() and m.name.endswith(".RS.flx"))
        for member in members:
            time_utc = timestamp_from_grib_member(member.name)
            if time_utc < START or time_utc > END:
                continue
            tmp_path = Path(tmp_dir) / Path(member.name).name
            with tf.extractfile(member) as src, tmp_path.open("wb") as dst:
                shutil.copyfileobj(src, dst)
            row = {"time_utc": time_utc, "source_member": member.name}
            row.update(nearest_values_from_grib_file(tmp_path))
            rows.append(row)
            tmp_path.unlink(missing_ok=True)
    return rows

rows = []
with tempfile.TemporaryDirectory(prefix="narr_gothic_point_") as tmp_dir:
    for i, row in enumerate(point_tars.itertuples(index=False), start=1):
        tar_path, is_cached = get_tar_path(row)
        print(f"{i:02d}/{len(point_tars)} {Path(tar_path).name}")
        rows.extend(extract_point_from_tar(tar_path, tmp_dir))
        if is_cached and not KEEP_DOWNLOADED_TARS:
            Path(tar_path).unlink(missing_ok=True)

forcing = pd.DataFrame(rows).sort_values("time_utc").reset_index(drop=True)
forcing_path = FORCING_DIR / f"narrflx_{TARGET_NAME}_wy{WY}_point_forcing.csv"
forcing.to_csv(forcing_path, index=False)

print(forcing_path)
print(forcing.shape)
forcing.head()


## Quick checks

Verify that the extracted cell and time coverage look right before converting the point CSV into the exact Noah-MP driver format you want.


In [ ]:
summary = {
    "first_time": forcing.time_utc.min(),
    "last_time": forcing.time_utc.max(),
    "n_timesteps": len(forcing),
    "expected_3hourly_timesteps": len(pd.date_range(START, END, freq="3h")),
    "narr_lat": forcing.narr_lat.dropna().iloc[0] if forcing.narr_lat.notna().any() else np.nan,
    "narr_lon": forcing.narr_lon.dropna().iloc[0] if forcing.narr_lon.notna().any() else np.nan,
    "missing_by_field": forcing.isna().sum().to_dict(),
}
summary


## Noah-MP/HRLDAS handoff

There is no `noahmp` or `hrldas.exe` executable on the current `PATH` in this environment. This cell creates a run directory and placeholder namelist; set `HRLDAS_EXE` to your compiled executable and adapt the forcing conversion to that build's required point or gridded input format.


In [ ]:
HRLDAS_EXE = os.environ.get("HRLDAS_EXE", "")

namelist = RUN_DIR / "namelist.hrldas"
namelist.write_text(
    f"""&NOAHLSM_OFFLINE
 START_YEAR  = {START.year},
 START_MONTH = {START.month},
 START_DAY   = {START.day},
 START_HOUR  = {START.hour},
 KHOUR = {int((END - START) / pd.Timedelta(hours=1)) + 1},
 FORCING_TIMESTEP = 10800,
 NOAH_TIMESTEP = 3600,
 OUTPUT_TIMESTEP = 3600,
/
"""
)

print(namelist)
print(namelist.read_text())

if not HRLDAS_EXE:
    print("Set HRLDAS_EXE=/path/to/hrldas.exe before running Noah-MP.")
elif not Path(HRLDAS_EXE).exists():
    raise FileNotFoundError(HRLDAS_EXE)
else:
    result = subprocess.run([HRLDAS_EXE], cwd=RUN_DIR, check=False, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
